# 02 — Data Inventory
## What is on disk, what is pending, what is synthetic

This notebook inspects available data files. It does not process them.
All processing happens in notebooks 03–05.

---

In [ ]:
from pathlib import Path
import os

BASE = Path('/media/rendier/0123-4567/bullet_cluster')

EXPECTED = {
    # X-ray
    'Chandra ObsID 554  evt2':     BASE / 'xray/chandra/acisf00554N004_evt2.fits.gz',
    'Chandra ObsID 3184 evt2':     BASE / 'xray/chandra/acisf03184N004_evt2.fits.gz',
    'Chandra ObsID 4984 evt2':     BASE / 'xray/chandra/acisf04984N004_evt2.fits.gz',
    'Chandra ObsID 5355 evt2':     BASE / 'xray/chandra/acisf05355N004_evt2.fits.gz',
    # Radio catalogue
    'MGCLS Table2 compact':        BASE / 'radio/meerkat/MGCLS_Table2_compact_catalogue.fits',
    'MGCLS Table4 diffuse':        BASE / 'radio/meerkat/MGCLS_Table4_diffuse.fits',
    # Radio cubes (SARAO gated)
    'MGCLS Stokes Q cube (REAL)':  BASE / 'radio/meerkat/MGCLS_1E0657-558_Q_cube.fits',
    'MGCLS Stokes U cube (REAL)':  BASE / 'radio/meerkat/MGCLS_1E0657-558_U_cube.fits',
    # Radio cubes (synthetic)
    'Synthetic Q (wave)':          BASE / 'radio/meerkat/synthetic/synthetic_Q_wave.fits',
    'Synthetic Q (particle)':      BASE / 'radio/meerkat/synthetic/synthetic_Q_particle.fits',
    # Planck SZ
    'Planck SZ y-map tgz':        BASE / 'mm_sz/planck/COM_CompMap_Compton-SZMap_R2.01.tgz',
    # Viewer layers
    'Layer: X-ray Chandra':        BASE / 'viewer/layers/xray_chandra.png',
    'Layer: Lensing κ':            BASE / 'viewer/layers/lensing_kappa.png',
    'Layer: SZ Planck':            BASE / 'viewer/layers/sz_planck.png',
    'Layer: Radio':                BASE / 'viewer/layers/radio_meerkat.png',
    'Layer: Optical JWST':         BASE / 'viewer/layers/optical_jwst.png',
}

print(f'DATA INVENTORY — {BASE}')
print('='*70)
n_present, n_missing, n_synthetic = 0, 0, 0
for label, path in EXPECTED.items():
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        tag = 'SYNTHETIC' if 'synthetic' in str(path).lower() else 'PRESENT'
        print(f'  {tag:10s}  {size_mb:8.1f} MB  {label}')
        if tag == 'SYNTHETIC': n_synthetic += 1
        else: n_present += 1
    else:
        print(f'  {"MISSING":10s}  {"":>8s}     {label}')
        n_missing += 1

print('='*70)
print(f'  Present: {n_present}   Synthetic: {n_synthetic}   Missing: {n_missing}')

In [ ]:
# Inspect Chandra observation summary
import gzip
from astropy.io import fits

chandra_dir = BASE / 'xray' / 'chandra'
obsids = [554, 3184, 4984, 4985, 4986, 5355, 5356, 5357, 5358, 5361]

print('Chandra ACIS-I event files:')
print(f'{"ObsID":>8}  {"Photons":>10}  {"Exposure (ks)":>15}  {"Status"}')
print('-'*55)
total_photons = 0
for obsid in obsids:
    pat = list(chandra_dir.glob(f'acisf{obsid:05d}*evt2.fits.gz'))
    if not pat:
        pat = list(chandra_dir.glob(f'acisf*{obsid}*evt2.fits.gz'))
    if not pat:
        print(f'{obsid:>8}  {"":>10}  {"":>15}  MISSING')
        continue
    try:
        with fits.open(pat[0]) as h:
            n = len(h['EVENTS'].data)
            exp = h['EVENTS'].header.get('EXPOSURE', 0) / 1000
        total_photons += n
        print(f'{obsid:>8}  {n:>10,}  {exp:>14.1f}  OK')
    except Exception as e:
        print(f'{obsid:>8}  ERROR: {e}')
print('-'*55)
print(f'  Total photons (0.5–7 keV): {total_photons:,}')

In [ ]:
# Inspect MGCLS diffuse catalogue
import numpy as np

cat_path = BASE / 'radio/meerkat/MGCLS_Table4_diffuse.fits'
if cat_path.exists():
    with fits.open(cat_path) as h:
        cat = h[1].data
        print(f'MGCLS Table4 diffuse: {len(cat)} sources')
        print(f'Columns: {list(cat.dtype.names)}')
    # Find Bullet Cluster sources
    ra_col  = next((c for c in cat.dtype.names if 'R' in c.upper() and 'A' in c.upper() and '.' in c), 'RA')
    dec_col = next((c for c in cat.dtype.names if 'D' in c.upper() and 'E' in c.upper() and 'C' in c.upper()), 'DEC')
    ra_arr  = cat[ra_col]
    dec_arr = cat[dec_col]
    mask = ((ra_arr > 103.5) & (ra_arr < 105.5) &
            (dec_arr > -56.5) & (dec_arr < -55.5))
    print(f'\nSources within 1° of cluster centre: {mask.sum()}')
    if mask.sum() > 0:
        for row in cat[mask]:
            print(f'  {row}')
else:
    print('MGCLS Table4 not present')

In [ ]:
# Check synthetic cubes if present, else summarise what will be generated
synth_dir = BASE / 'radio/meerkat/synthetic'

if synth_dir.exists():
    print('Synthetic cube directory contents:')
    for f in sorted(synth_dir.iterdir()):
        size_mb = f.stat().st_size / 1e6
        print(f'  {size_mb:8.1f} MB  {f.name}')
else:
    print('Synthetic directory not yet created.')
    print('Run notebook 03_radio_synthesis.ipynb to generate synthetic cubes.')
    print()
    print('Expected files after generation:')
    print('  synthetic_I_wave.fits    — Stokes I (wave DM model)')
    print('  synthetic_Q_wave.fits    — Stokes Q')
    print('  synthetic_U_wave.fits    — Stokes U')
    print('  synthetic_freqs_wave.dat — frequency axis (Hz)')
    print('  synthetic_I_particle.fits  (particle DM model)')
    print('  synthetic_Q_particle.fits')
    print('  synthetic_U_particle.fits')
    print('  Cube dimensions: 800 freq × 256 × 256 pixels, ~900 MB each set')